In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import os 
from padelpy import padeldescriptor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from rdkit.Chem import rdFingerprintGenerator  
import pandas as pd 
from padelpy import from_smiles
# import warnings
# warnings.filterwarnings('ignore')


In [19]:
df = pd.read_excel("FINAL.xlsx")
df.head()


,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,Unnamed: 6,ck_swis,MW_swis,ESOL Class,...,Unnamed: 12,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
0,1,145991278.0,C1[C@]2(C3=CC(=NC=C3OC4=C2C=C(C=C4)C5=NC(=CC=C...,443.4,443,True,NaN,443.0,443.41,Moderately soluble,...,NaN,443.0,443.12,-4.881416,0.000000,0.001933,0.909164,0.717378,0.947849,0.989082
1,2,145990468.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,463.5,463,True,NaN,463.0,463.46,Moderately soluble,...,NaN,463.0,463.14,-4.828875,0.000000,0.000030,0.942825,0.776856,0.794406,0.988022
2,3,122172743.0,CN1C(NC2(C1=O)C3=C(C=CC(=C3)OCC(F)(F)F)OC24CCC...,447.4,447,True,NaN,447.0,447.45,Moderately soluble,...,NaN,447.0,447.18,-5.399611,0.000046,0.969402,0.755374,0.757735,0.426495,0.804849
3,4,145983198.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,450.5,450,True,NaN,450.0,450.49,Moderately soluble,...,NaN,450.0,450.17,-4.864286,0.000006,0.004028,0.885521,0.696834,0.734198,0.973429
4,5,45254510.0,CC1(COC1)C#CC2=CC3=C(N=C2)OC4=C([C@@]35COC(=N5...,442.4,442,True,NaN,442.0,442.44,Moderately soluble,...,NaN,442.0,442.14,-4.885711,0.000002,0.698582,0.770069,0.598314,0.942510,0.976820


In [6]:
print(df.shape)
df.isnull().sum()

(3519, 22)


COUNT             0
cid             663
SMILES          663
MW_pub          663
ck pub            0
VERIFY            0
Unnamed: 6     3519
ck_swis        1519
MW_swis         663
ESOL Class      663
Unnamed: 10    3519
Unnamed: 11    3519
Unnamed: 12    3519
CK_AD           663
MW_AD           663
caco2           663
hia             663
BBB             663
hERG            663
hERG-10um       663
Ames            663
H-HT            663
dtype: int64

In [20]:
# Drop rows with null SMILES values
df = df.dropna(subset=['SMILES'])

# Verify the result
print(df.shape)
df.head()


(2856, 22)


,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,Unnamed: 6,ck_swis,MW_swis,ESOL Class,...,Unnamed: 12,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
0,1,145991278.0,C1[C@]2(C3=CC(=NC=C3OC4=C2C=C(C=C4)C5=NC(=CC=C...,443.4,443,True,NaN,443.0,443.41,Moderately soluble,...,NaN,443.0,443.12,-4.881416,0.000000,0.001933,0.909164,0.717378,0.947849,0.989082
1,2,145990468.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,463.5,463,True,NaN,463.0,463.46,Moderately soluble,...,NaN,463.0,463.14,-4.828875,0.000000,0.000030,0.942825,0.776856,0.794406,0.988022
2,3,122172743.0,CN1C(NC2(C1=O)C3=C(C=CC(=C3)OCC(F)(F)F)OC24CCC...,447.4,447,True,NaN,447.0,447.45,Moderately soluble,...,NaN,447.0,447.18,-5.399611,0.000046,0.969402,0.755374,0.757735,0.426495,0.804849
3,4,145983198.0,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,450.5,450,True,NaN,450.0,450.49,Moderately soluble,...,NaN,450.0,450.17,-4.864286,0.000006,0.004028,0.885521,0.696834,0.734198,0.973429
4,5,45254510.0,CC1(COC1)C#CC2=CC3=C(N=C2)OC4=C([C@@]35COC(=N5...,442.4,442,True,NaN,442.0,442.44,Moderately soluble,...,NaN,442.0,442.14,-4.885711,0.000002,0.698582,0.770069,0.598314,0.942510,0.976820


In [25]:
# Drop columns with more than 20% null values
threshold = 0.2  # 20% threshold
df = df.loc[:, df.isnull().mean() < threshold]

# Verify the result
print(df.shape)
print(df.columns)


(2856, 17)
Index(['COUNT', 'cid', 'SMILES', 'MW_pub', 'ck pub', 'VERIFY', 'MW_swis',
       'ESOL Class', 'CK_AD', 'MW_AD', 'caco2', 'hia', 'BBB', 'hERG',
       'hERG-10um', 'Ames', 'H-HT'],
      dtype='str')


In [28]:
df['SMILES'].value_counts()

SMILES
C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)NC(=O)C3=C(C=C(C=N3)C#N)Cl)F              6
C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C4=NC=C(C=C4)C#N)F                         5
CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C@]35COC(=N5)N)C6=CC(=NC=C6)F              4
CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C@]35COC(=N5)N)C6=CCCOC6                   4
C1CC1C2=NC=C3C(=C2)[C@]4(COC(=N4)N)C5=C(O3)C=CC(=C5)NC(=O)C6=NC=C(C=C6)Cl               4
                                                                                       ..
CCCC#CC1=CN=C(C=C1)/C(=C/C2=CC(=C(C=C2)F)[C@@]3([C@@H]4C[C@@]4(SC(=N3)N)C(=O)NC)C)/F    1
CC(C)N1C(NC2(C1=O)C3=C(CC24CCC5=CC=CC=C5CC4)C=CC(=C3)OCC6CC6)N                          1
CN1C(=O)C2(C3=C(CC24CCC5=CC=CC=C5CC4)C=CC(=C3)Br)N=C1N                                  1
COC1CCC2(CC1)CC3=C(C24C(=O)N(C(=N4)N)CC(F)(F)F)C=C(C=C3)C5=CN=CC(=C5)C#N                1
CC1=C(C=CC2=C1CCC3=C(C24C(=O)N(C(=N4)N)C)C=C(C=C3)C5=CN=CN=C5)OC                        1
Nam

In [31]:
# Locate the row(s) with the given SMILES string
target_smiles = "C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C4=NC=C(C=C4)C#N)F"

# Filter the dataframe
matched_rows = df[df['SMILES'] == target_smiles]

# Display the result
matched_rows


,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,MW_swis,ESOL Class,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
405,406,121298035.0,C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C...,389.4,389,True,389.39,Soluble,389.0,389.14,-4.984455,0.000109,0.015061,0.853415,0.580703,0.797988,0.989465
1509,1510,121298035.0,C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C...,389.4,389,True,389.39,Soluble,389.0,389.14,-4.984455,0.000109,0.015061,0.853415,0.580703,0.797988,0.989465
1956,1957,121298035.0,C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C...,389.4,389,True,389.39,Soluble,389.0,389.14,-4.984455,0.000109,0.015061,0.853415,0.580703,0.797988,0.989465
2125,2126,121298035.0,C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C...,389.4,389,False,389.39,Soluble,389.0,389.14,-4.984455,0.000109,0.015061,0.853415,0.580703,0.797988,0.989465
2526,2527,121298035.0,C[C@]1(CN2C=CN=C2C(=N1)N)C3=C(C=CC(=C3)NC(=O)C...,389.4,389,False,389.39,Soluble,389.0,389.14,-4.984455,0.000109,0.015061,0.853415,0.580703,0.797988,0.989465


In [29]:
# Locate the row(s) with the given SMILES string
target_smiles = "C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)NC(=O)C3=C(C=C(C=N3)C#N)Cl)F"

# Filter the dataframe
matched_rows = df[df['SMILES'] == target_smiles]

# Display the result
matched_rows


,COUNT,cid,SMILES,MW_pub,ck pub,VERIFY,MW_swis,ESOL Class,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
702,703,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,True,456.78,Moderately soluble,456.0,456.07,-4.938309,0.000031,0.006264,0.856123,0.664289,0.779237,0.984255
1175,1176,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,True,456.78,Moderately soluble,456.0,456.07,390.892548,-4.938309,0.006264,0.856123,0.664289,0.779237,0.984255
1357,1358,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,True,456.78,Moderately soluble,456.0,456.07,-4.938309,0.000031,0.006264,0.856123,0.664289,0.779237,0.984255
1749,1750,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,True,456.78,Moderately soluble,456.0,456.07,-4.938309,0.000031,0.006264,0.856123,0.664289,0.779237,0.984255
2476,2477,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,False,456.78,Moderately soluble,456.0,456.07,-4.938309,0.000031,0.006264,0.856123,0.664289,0.779237,0.984255
2516,2517,78321445.0,C[C@]1(C[C@H](OC(=N1)N)C(F)(F)F)C2=C(C=CC(=N2)...,456.8,456,False,456.78,Moderately soluble,456.0,456.07,-4.938309,0.000031,0.006264,0.856123,0.664289,0.779237,0.984255


In [30]:
df['ESOL Class'].value_counts()

ESOL Class
Moderately soluble    1665
Soluble               1189
Poorly soluble           2
Name: count, dtype: int64

In [33]:
# Drop the 'cid' column
df = df.drop(columns=['cid'])

# Verify the result
print(df.columns)


Index(['COUNT', 'SMILES', 'MW_pub', 'ck pub', 'VERIFY', 'MW_swis',
       'ESOL Class', 'CK_AD', 'MW_AD', 'caco2', 'hia', 'BBB', 'hERG',
       'hERG-10um', 'Ames', 'H-HT'],
      dtype='str')


In [34]:
df.head()

,COUNT,SMILES,MW_pub,ck pub,VERIFY,MW_swis,ESOL Class,CK_AD,MW_AD,caco2,hia,BBB,hERG,hERG-10um,Ames,H-HT
0,1,C1[C@]2(C3=CC(=NC=C3OC4=C2C=C(C=C4)C5=NC(=CC=C...,443.4,443,True,443.41,Moderately soluble,443.0,443.12,-4.881416,0.000000,0.001933,0.909164,0.717378,0.947849,0.989082
1,2,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,463.5,463,True,463.46,Moderately soluble,463.0,463.14,-4.828875,0.000000,0.000030,0.942825,0.776856,0.794406,0.988022
2,3,CN1C(NC2(C1=O)C3=C(C=CC(=C3)OCC(F)(F)F)OC24CCC...,447.4,447,True,447.45,Moderately soluble,447.0,447.18,-5.399611,0.000046,0.969402,0.755374,0.757735,0.426495,0.804849
3,4,CC#CC1=CC(=CN=C1)C2=CC3=C(C=C2)OC4=CN=C(C=C4[C...,450.5,450,True,450.49,Moderately soluble,450.0,450.17,-4.864286,0.000006,0.004028,0.885521,0.696834,0.734198,0.973429
4,5,CC1(COC1)C#CC2=CC3=C(N=C2)OC4=C([C@@]35COC(=N5...,442.4,442,True,442.44,Moderately soluble,442.0,442.14,-4.885711,0.000002,0.698582,0.770069,0.598314,0.942510,0.976820


In [35]:
import numpy as np

# Define aggregation rules for numeric columns
numeric_cols = ['MW_pub', 'BBB', 'hERG', 'Ames', 'H-HT']  # adjust as needed

# Group by SMILES + VERIFY + ESOL Class
df_grouped = (
    df.groupby(['SMILES', 'VERIFY', 'ESOL Class'], as_index=False)[numeric_cols]
      .mean()
)

# Verify the result
print(df_grouped.head())
print(df_grouped.shape)


                                              SMILES  VERIFY  \
0  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   False   
1  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...   False   
2  B1(OC(C(O1)(C)C)(C)C)C2=CC3=C(CC4(C35C(=O)N(C(...    True   
3  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C(=C2)F)F)[C@...    True   
4  C#CCOC1=CN=C(C=C1)C(=O)NC2=CC(=C(C=C2)F)[C@@]3...   False   

           ESOL Class  MW_pub       BBB      hERG      Ames      H-HT  
0  Moderately soluble   468.4  0.982377  0.138762  0.684111  0.349169  
1  Moderately soluble   476.3  0.999029  0.262471  0.351038  0.586856  
2  Moderately soluble   472.4  0.989686  0.112066  0.887385  0.556827  
3  Moderately soluble   486.4  0.001523  0.666432  0.854345  0.977018  
4             Soluble   430.4  0.004441  0.332171  0.871076  0.912706  
(1860, 8)


In [38]:
df_grouped['ESOL Class'].value_counts()

ESOL Class
Moderately soluble    1070
Soluble                788
Poorly soluble           2
Name: count, dtype: int64

In [40]:
df_grouped['SMILES'].duplicated().sum()

596

In [41]:
df_grouped.to_csv('grouped_data.csv', index=False)
